# Experiments 33-36 — Umayya ibn Abi al-Salt (Scraped Fresh)

He's not in your existing 260-poet database, but he's on the same
source site (aldiwan.net) that database was originally built from, at
`cat-poet-umaiya-ibn-abi-Salt`. This notebook scrapes his poems
directly, embeds him using the exact same method used for all 260
other poets (verse → poem → poet hierarchical average, same model,
same settings), then runs the four comparisons.

**This notebook does not need `poems.db`.** Umayya's embedding is built
fresh from his own scraped poems; the Quranic side reuses your cached
surah/hizb/hizb-pair vectors if present, or fetches and embeds them
fresh if not.

**Scraping is polite** — same delay between requests as the original
scraper, so it won't hammer the site. He has a modest number of poems,
so this should only take a minute or two.

Run cells top to bottom, **Shift+Enter**.

In [ ]:
# CELL 1 -- Install packages
!pip -q install sentence-transformers torch scikit-learn umap-learn hdbscan pandas numpy matplotlib seaborn requests beautifulsoup4 lxml

In [ ]:
# CELL 2 -- Configuration
import re, json, warnings, pickle, time
from pathlib import Path
from urllib.parse import urljoin

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from bs4 import BeautifulSoup

warnings.filterwarnings("ignore")

CACHE_DIR = Path("quran_cache"); CACHE_DIR.mkdir(exist_ok=True)
EMBED_CACHE_DIR = Path("embed_cache"); EMBED_CACHE_DIR.mkdir(exist_ok=True)
FIGURES_DIR = Path("output/figures"); FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR = Path("output/tables"); TABLES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = Path("output/reports"); REPORTS_DIR.mkdir(parents=True, exist_ok=True)

SBERT_MODEL_NAME = "akhooli/Arabic-SBERT-100K"
MAX_VERSES_PER_POEM = 20

UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1
UMAP_N_COMPONENTS_HIGH = 50
UMAP_METRIC = "cosine"
HDBSCAN_MIN_CLUSTER_SIZE = 5
HDBSCAN_MIN_SAMPLES = 3

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
import torch
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    print("GPU available:", torch.cuda.get_device_name(0))
else:
    print("No GPU found -- will run on CPU.")

def split_verses(poem_text):
    if not poem_text or not isinstance(poem_text, str):
        return []
    verses = re.split(r'[\n\r]+|[.!\u061F?\u061B;]+', poem_text)
    return [v.strip() for v in verses if len(v.strip()) > 10]

def normalize_word(w):
    w = re.sub(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06ED\u0670]", "", w)
    w = re.sub(r"\u0640", "", w)
    w = re.sub(r"[\u0622\u0623\u0625\u0671]", "\u0627", w)
    return w.strip()

def is_valid_arabic_verse(text, min_chars=3):
    if not text or len(text.strip()) < min_chars:
        return False
    return len(re.findall(r"[\u0600-\u06FF]", text)) >= min_chars

ALDIWAN_BASE = "https://www.aldiwan.net"
SCRAPER_DELAY = 1.5
SCRAPER_HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 300, "savefig.bbox": "tight", "font.size": 11})
print("Config loaded.")

In [ ]:
# CELL 3 -- Scrape Umayya ibn Abi al-Salt's poems (same method as the
# original 260-poet corpus, same source site, same politeness delay)
session = requests.Session()
session.headers.update(SCRAPER_HEADERS)

def fetch_soup(url):
    resp = session.get(url, timeout=20)
    resp.raise_for_status()
    time.sleep(SCRAPER_DELAY)
    return BeautifulSoup(resp.text, "lxml")

POET_URL = f"{ALDIWAN_BASE}/cat-poet-umaiya-ibn-abi-Salt"
print(f"Fetching poet page: {POET_URL}")
poet_soup = fetch_soup(POET_URL)

poem_links, seen = [], set()
for a in poet_soup.find_all("a", href=True):
    href = a["href"]
    m = re.search(r"/poem(\d+)\.html", href)
    if not m:
        continue
    url = urljoin(ALDIWAN_BASE, href)
    title = a.get_text(strip=True)
    if title and url not in seen:
        seen.add(url)
        poem_links.append({"title": title, "url": url})

print(f"Found {len(poem_links)} poems.")

umayya_poems = []
for i, poem in enumerate(poem_links, 1):
    try:
        psoup = fetch_soup(poem["url"])
    except requests.RequestException as e:
        print(f"  Skipped (fetch error): {poem['url']}")
        continue
    verse_tags = psoup.select("h3")
    verses = [t.get_text(strip=True) for t in verse_tags]
    verses = [v for v in verses if is_valid_arabic_verse(v)]
    if len(verses) < 2:
        continue
    poem_text = "\n".join(verses)
    umayya_poems.append(poem_text)
    if i % 10 == 0 or i == len(poem_links):
        print(f"  {i}/{len(poem_links)} poems processed")

print(f"\nScraped {len(umayya_poems)} usable poems.")
total_verses = sum(len(split_verses(p)) for p in umayya_poems)
print(f"Total verses: {total_verses}")

if len(umayya_poems) == 0:
    raise SystemExit("No poems were successfully scraped -- check the URL/site is reachable "
                     "and try again before continuing.")

In [ ]:
# CELL 4 -- Embed Umayya (same method, same settings as all 260 other poets)
from sentence_transformers import SentenceTransformer

print("Loading model:", SBERT_MODEL_NAME)
model = SentenceTransformer(SBERT_MODEL_NAME)
print("Loaded.")

def _encode(texts, batch_size=64):
    if not texts:
        return np.array([])
    return model.encode(texts, batch_size=batch_size, show_progress_bar=False,
                         normalize_embeddings=True, convert_to_numpy=True)

def embed_poem_verse_average_single(poems, max_verses=MAX_VERSES_PER_POEM, seed=RANDOM_SEED):
    # Identical logic to the function used for the original 260 poets --
    # verse -> poem -> poet hierarchical mean.
    rng = np.random.RandomState(seed)
    poem_vectors = []
    for poem in poems:
        verses = split_verses(poem)
        if len(verses) > max_verses:
            idx = rng.choice(len(verses), max_verses, replace=False)
            verses = [verses[i] for i in sorted(idx)]
        if verses:
            poem_vectors.append(np.mean(_encode(verses), axis=0))
    return np.mean(poem_vectors, axis=0) if poem_vectors else None

umayya_name = "Umayya ibn Abi al-Salt"
umayya_vector = embed_poem_verse_average_single(umayya_poems)
umayya_poem_count = len(umayya_poems)
umayya_verse_count = total_verses

print(f"Umayya embedded from {umayya_poem_count} poems, {umayya_verse_count} verses.")

In [ ]:
# CELL 5 -- Load Quran text (cached if available) and build unit vectors
cache_file = CACHE_DIR / "quran_ayat_with_hizb.json"
if cache_file.exists():
    print("Loading Quran (with hizb metadata) from local cache...")
    with open(cache_file, encoding="utf-8") as f:
        surahs_raw = json.load(f)
else:
    print("Fetching Quran from Al Quran Cloud API...")
    resp = requests.get("https://api.alquran.cloud/v1/quran/quran-uthmani", timeout=60)
    resp.raise_for_status()
    surahs_raw = resp.json()["data"]["surahs"]
    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(surahs_raw, f, ensure_ascii=False)
    print("Fetched and cached.")

surah_names = {}
ayah_records = []
for s in surahs_raw:
    snum = s["number"]
    surah_names[snum] = s["englishName"]
    for a in s["ayahs"]:
        hizb_number = ((a["hizbQuarter"] - 1) // 4) + 1
        ayah_records.append({"surah_number": snum, "ayah_number": a["numberInSurah"],
                             "text": a["text"], "hizb_number": hizb_number})
ayah_df = pd.DataFrame(ayah_records)
print(f"Surahs: {ayah_df['surah_number'].nunique()} | Ayat: {len(ayah_df)}")

ayah_embed_cache_file = EMBED_CACHE_DIR / "ayah_embeddings.pkl"
if ayah_embed_cache_file.exists():
    print("Loading cached ayah embeddings...")
    with open(ayah_embed_cache_file, "rb") as f:
        ayah_embeddings = pickle.load(f)
    print(f"Loaded {len(ayah_embeddings)} cached ayah embeddings.")
else:
    print(f"Embedding all {len(ayah_df)} ayat individually (slow, one-time only)...")
    all_texts = ayah_df["text"].tolist()
    all_vecs = _encode(all_texts, batch_size=64)
    ayah_embeddings = {}
    for (snum, anum), vec in zip(zip(ayah_df["surah_number"], ayah_df["ayah_number"]), all_vecs):
        ayah_embeddings[(snum, anum)] = vec
    with open(ayah_embed_cache_file, "wb") as f:
        pickle.dump(ayah_embeddings, f)
    print(f"Done and cached. {len(ayah_embeddings)} ayat embedded.")

def vector_for_ayat(ayat_keys):
    vecs = [ayah_embeddings[k] for k in ayat_keys if k in ayah_embeddings]
    return np.mean(vecs, axis=0) if vecs else None

surah_vectors = {}
for snum in surah_names:
    keys = list(zip(ayah_df[ayah_df["surah_number"] == snum]["surah_number"],
                    ayah_df[ayah_df["surah_number"] == snum]["ayah_number"]))
    v = vector_for_ayat(keys)
    if v is not None:
        surah_vectors[snum] = v

hizb_vectors = {}
for hizb_num, group in ayah_df.groupby("hizb_number"):
    keys = list(zip(group["surah_number"], group["ayah_number"]))
    v = vector_for_ayat(keys)
    if v is not None:
        hizb_vectors[hizb_num] = v

two_hizb_vectors = {}
for i in range(1, 61, 2):
    sub = ayah_df[ayah_df["hizb_number"].isin([i, i+1])]
    keys = list(zip(sub["surah_number"], sub["ayah_number"]))
    v = vector_for_ayat(keys)
    if v is not None:
        two_hizb_vectors[f"{i}-{i+1}"] = v

quran_whole_vector = np.mean(list(surah_vectors.values()), axis=0)
print(f"\nSurah vectors: {len(surah_vectors)} | Hizb vectors: {len(hizb_vectors)} | "
      f"Hizb-pair vectors: {len(two_hizb_vectors)}")

In [ ]:
# CELL 6 -- Shared clustering function
import umap, hdbscan
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score

def run_umap(matrix, n_components, n_neighbors, min_dist, metric="cosine", random_state=RANDOM_SEED):
    reducer = umap.UMAP(n_neighbors=min(n_neighbors, len(matrix) - 1), n_components=n_components,
                         min_dist=min_dist, metric=metric, random_state=random_state)
    return reducer.fit_transform(matrix)

def cluster_and_report(embeddings_dict, label_types, title, n_neighbors=UMAP_N_NEIGHBORS,
                       min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE, verbose=True):
    names = sorted(embeddings_dict.keys())
    n = len(names)
    emb_matrix = np.array([embeddings_dict[nm] for nm in names])
    umap_high = run_umap(emb_matrix, min(UMAP_N_COMPONENTS_HIGH, max(2, n - 2)), n_neighbors, 0.0, UMAP_METRIC)
    clusterer = hdbscan.HDBSCAN(min_cluster_size=min(min_cluster_size, max(2, n // 10)),
                                min_samples=HDBSCAN_MIN_SAMPLES,
                                cluster_selection_method="eom", metric="euclidean")
    labels = clusterer.fit_predict(umap_high)
    umap_2d = run_umap(emb_matrix, 2, n_neighbors, UMAP_MIN_DIST, UMAP_METRIC)
    mask = labels >= 0
    n_clusters = len(set(labels[mask])) if mask.sum() > 0 else 0
    n_outliers = int(np.sum(labels == -1))
    sil = float(silhouette_score(umap_high[mask], labels[mask])) if n_clusters >= 2 and mask.sum() > n_clusters else None
    result_df = pd.DataFrame({"name": names, "type": [label_types[nm] for nm in names],
        "cluster": labels, "umap_x": umap_2d[:, 0], "umap_y": umap_2d[:, 1]})
    if verbose:
        print(f"=== {title} ===")
        print(f"Total: {n} | Clusters: {n_clusters} | Outliers: {n_outliers} | Silhouette: {sil}")
    return result_df, {"n_entities": n, "n_clusters": n_clusters, "n_outliers": n_outliers, "silhouette": sil}

print("Clustering helper ready.")

## Experiments 33-35 — Umayya vs. Surahs / Hizb / Hizb-Pairs

In [ ]:
# CELL 7 -- Experiments 33, 34, 35: cluster Umayya against surahs/hizb/pairs,
# plus full ranked similarity breakdown for each
def cluster_lone_poet(poet_name, poet_vector, units_dict, unit_label, exp_num):
    embeddings = {poet_name: poet_vector, **units_dict}
    types = {nm: ("Poet" if nm == poet_name else unit_label) for nm in embeddings}
    result_df, metrics = cluster_and_report(embeddings, types, f"Experiment {exp_num}: Umayya vs {unit_label}")
    poet_row = result_df[result_df["name"] == poet_name].iloc[0]
    status = "OUTLIER" if poet_row["cluster"] == -1 else f"Cluster {poet_row['cluster']}"
    same_cluster = result_df[(result_df["cluster"] == poet_row["cluster"]) & (result_df["name"] != poet_name)]
    print(f"Umayya landed in: {status}")
    print(f"Units sharing that cluster: {len(same_cluster)}")
    if len(same_cluster) > 0 and len(same_cluster) <= 20:
        print("  " + "; ".join(same_cluster["name"].tolist()))
    return result_df, metrics

def similarity_breakdown(poet_vector, units_dict):
    names = sorted(units_dict.keys())
    vecs = np.array([units_dict[n] for n in names])
    sims = cosine_similarity([poet_vector], vecs)[0]
    df = pd.DataFrame({"name": names, "cosine_similarity": sims}).sort_values(
        "cosine_similarity", ascending=False).reset_index(drop=True)
    return df

# --- Experiment 33: vs 114 surahs ---
surah_units = {f"Surah {n}: {surah_names[n]}": v for n, v in surah_vectors.items()}
exp33_df, exp33_metrics = cluster_lone_poet(umayya_name, umayya_vector, surah_units, "Surah", 33)
exp33_df.to_csv(TABLES_DIR / "experiment33_clusters.csv", index=False)

exp33_sim = similarity_breakdown(umayya_vector, surah_units)
exp33_sim.to_csv(TABLES_DIR / "experiment33_similarity_breakdown.csv", index=False)
print(f"\nTop 10 most similar surahs to Umayya:")
print(exp33_sim.head(10).to_string(index=False))
print(f"\nBottom 5 least similar surahs to Umayya:")
print(exp33_sim.tail(5).to_string(index=False))
print(f"\nSimilarity stats: mean={exp33_sim['cosine_similarity'].mean():.4f}, "
      f"median={exp33_sim['cosine_similarity'].median():.4f}, "
      f"min={exp33_sim['cosine_similarity'].min():.4f}, max={exp33_sim['cosine_similarity'].max():.4f}")

# --- Experiment 34: vs 60 hizb ---
print("\n" + "=" * 70)
hizb_units = {f"Hizb {n}": v for n, v in hizb_vectors.items()}
exp34_df, exp34_metrics = cluster_lone_poet(umayya_name, umayya_vector, hizb_units, "Hizb", 34)
exp34_df.to_csv(TABLES_DIR / "experiment34_clusters.csv", index=False)

exp34_sim = similarity_breakdown(umayya_vector, hizb_units)
exp34_sim.to_csv(TABLES_DIR / "experiment34_similarity_breakdown.csv", index=False)
print(f"\nTop 10 most similar hizb to Umayya:")
print(exp34_sim.head(10).to_string(index=False))
print(f"\nSimilarity stats: mean={exp34_sim['cosine_similarity'].mean():.4f}, "
      f"median={exp34_sim['cosine_similarity'].median():.4f}")

# --- Experiment 35: vs 30 hizb-pairs ---
print("\n" + "=" * 70)
hizbpair_units = {f"Hizb-pair {label}": v for label, v in two_hizb_vectors.items()}
exp35_df, exp35_metrics = cluster_lone_poet(umayya_name, umayya_vector, hizbpair_units, "HizbPair", 35)
exp35_df.to_csv(TABLES_DIR / "experiment35_clusters.csv", index=False)

exp35_sim = similarity_breakdown(umayya_vector, hizbpair_units)
exp35_sim.to_csv(TABLES_DIR / "experiment35_similarity_breakdown.csv", index=False)
print(f"\nTop 10 most similar hizb-pairs to Umayya:")
print(exp35_sim.head(10).to_string(index=False))
print(f"\nSimilarity stats: mean={exp35_sim['cosine_similarity'].mean():.4f}, "
      f"median={exp35_sim['cosine_similarity'].median():.4f}")

## Experiment 36 — Umayya vs. the Whole Quran (Direct Similarity)

Only 2 entities, so this is direct cosine similarity, not clustering.

In [ ]:
# CELL 8 -- Experiment 36: Umayya vs whole Quran, direct similarity
sim_36 = float(cosine_similarity([umayya_vector], [quran_whole_vector])[0][0])
print(f"Experiment 36: Umayya vs Quran (whole)")
print(f"Cosine similarity: {sim_36:.4f}")
print("\nFor comparison (already-established reference values from earlier work):")
print(f"  Poetry corpus's own average poet-to-poet similarity: 0.799")
print(f"  35Vector (35 well-documented poets) vs whole Quran: 0.962")
print(f"  AllPoets (all 260 poets) vs whole Quran: 0.954")

with open(REPORTS_DIR / "experiment36_result.json", "w") as f:
    json.dump({"umayya_vs_quran_whole_similarity": sim_36}, f, indent=2)

In [ ]:
# CELL 9 -- Figures
def plot_umayya(df, title, save_name, unit_marker):
    fig, ax = plt.subplots(figsize=(12, 9))
    unique_clusters = sorted(df["cluster"].unique())
    n_clust_plot = len([c for c in unique_clusters if c >= 0])
    colors = plt.cm.tab20(np.linspace(0, 1, max(n_clust_plot, 1)))
    for cl in unique_clusters:
        sub = df[df["cluster"] == cl]
        color = "gray" if cl == -1 else colors[cl % len(colors)]
        units = sub[sub["name"] != umayya_name]
        poet = sub[sub["name"] == umayya_name]
        if len(units) > 0:
            ax.scatter(units["umap_x"], units["umap_y"], c=[color], marker=unit_marker, s=80,
                      alpha=0.8, edgecolors="black", linewidth=0.4)
        if len(poet) > 0:
            ax.scatter(poet["umap_x"], poet["umap_y"], c="red", marker="*", s=550,
                      edgecolors="black", linewidth=1.5, zorder=10, label="Umayya ibn Abi al-Salt")
    ax.set_xlabel("UMAP Dimension 1"); ax.set_ylabel("UMAP Dimension 2")
    ax.set_title(title)
    ax.legend(loc="best")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / save_name, dpi=300, bbox_inches="tight")
    plt.show()

plot_umayya(exp33_df, "Experiment 33: Umayya vs 114 Surahs", "experiment33_umap.png", "^")
plot_umayya(exp34_df, "Experiment 34: Umayya vs 60 Hizb", "experiment34_umap.png", "D")
plot_umayya(exp35_df, "Experiment 35: Umayya vs 30 Hizb-Pairs", "experiment35_umap.png", "s")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, sim_df, title in [(axes[0], exp33_sim, "vs 114 Surahs"),
                          (axes[1], exp34_sim, "vs 60 Hizb"),
                          (axes[2], exp35_sim, "vs 30 Hizb-Pairs")]:
    ax.hist(sim_df["cosine_similarity"], bins=20, color="#3498db", edgecolor="black")
    ax.axvline(sim_df["cosine_similarity"].mean(), color="red", linestyle="--",
              label=f"mean={sim_df['cosine_similarity'].mean():.3f}")
    ax.set_xlabel("Cosine similarity"); ax.set_ylabel("Count")
    ax.set_title(title)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "experiment33_34_35_similarity_distributions.png", dpi=300, bbox_inches="tight")
plt.show()

print("All figures saved.")

In [ ]:
# CELL 10 -- Final report
report_path = REPORTS_DIR / "experiments_33_36_report.txt"
with open(report_path, "w", encoding="utf-8") as f:
    f.write("=" * 70 + "\n")
    f.write(f"EXPERIMENTS 33-36: {umayya_name} VS THE QURAN\n")
    f.write("=" * 70 + "\n\n")

    f.write(f"Poet: {umayya_name} (scraped fresh from aldiwan.net)\n")
    f.write(f"Poems: {umayya_poem_count} | Total verses: {umayya_verse_count}\n\n")

    f.write("EXPERIMENT 33: VS 114 SURAHS\n" + "-" * 40 + "\n")
    for k, v in exp33_metrics.items():
        f.write(f"  {k}: {v}\n")
    f.write("  Top 10 most similar surahs:\n")
    f.write(exp33_sim.head(10).to_string(index=False) + "\n")
    f.write(f"  Similarity stats: mean={exp33_sim['cosine_similarity'].mean():.4f}, "
            f"median={exp33_sim['cosine_similarity'].median():.4f}\n")

    f.write("\nEXPERIMENT 34: VS 60 HIZB\n" + "-" * 40 + "\n")
    for k, v in exp34_metrics.items():
        f.write(f"  {k}: {v}\n")
    f.write("  Top 10 most similar hizb:\n")
    f.write(exp34_sim.head(10).to_string(index=False) + "\n")
    f.write(f"  Similarity stats: mean={exp34_sim['cosine_similarity'].mean():.4f}, "
            f"median={exp34_sim['cosine_similarity'].median():.4f}\n")

    f.write("\nEXPERIMENT 35: VS 30 HIZB-PAIRS\n" + "-" * 40 + "\n")
    for k, v in exp35_metrics.items():
        f.write(f"  {k}: {v}\n")
    f.write("  Top 10 most similar hizb-pairs:\n")
    f.write(exp35_sim.head(10).to_string(index=False) + "\n")
    f.write(f"  Similarity stats: mean={exp35_sim['cosine_similarity'].mean():.4f}, "
            f"median={exp35_sim['cosine_similarity'].median():.4f}\n")

    f.write("\nEXPERIMENT 36: VS WHOLE QURAN (DIRECT SIMILARITY)\n" + "-" * 40 + "\n")
    f.write(f"  Cosine similarity: {sim_36:.4f}\n")
    f.write(f"  For comparison -- poetry corpus's own average: 0.799\n")
    f.write(f"  For comparison -- 35Vector vs whole Quran: 0.962\n")
    f.write(f"  For comparison -- AllPoets vs whole Quran: 0.954\n")

    f.write("\nHOW TO READ THIS\n" + "-" * 40 + "\n")
    f.write("  Cluster membership for a single lone poet should be read\n")
    f.write("  cautiously -- a lone point tends to get pulled into whatever\n")
    f.write("  cluster is nearest almost by default, regardless of what it\n")
    f.write("  actually is (established via the synthetic-text control test\n")
    f.write("  earlier in this project). The similarity breakdown numbers\n")
    f.write("  (mean, median, and the full ranked list) are the more\n")
    f.write("  informative part of this result.\n")

print(f"Report written to {report_path}")
print()
print(open(report_path, encoding="utf-8").read())

## Done

Output in `output/`:
- `output/tables/experiment33/34/35_clusters.csv` — cluster membership
- `output/tables/experiment33/34/35_similarity_breakdown.csv` — full
  ranked similarity to every single surah/hizb/pair
- `output/reports/experiment36_result.json` — the whole-Quran number
- `output/figures/` — UMAP plots for 33-35 (Umayya marked as a red
  star), plus a similarity-distribution histogram for all three
- `output/reports/experiments_33_36_report.txt` — everything together

Send this back and I'll build the report.